In [11]:
# If needed, install packages in your environment first:
# !pip install -r lab4b_requirements.txt


## 2. Imports and helper functions

In [12]:
from __future__ import annotations
import argparse
import json
import math
import os
import random
import re
import statistics
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
import numpy as np
import pandas as pd
from tqdm import tqdm

def set_seed(seed: int = 42) -> None:
    """Make runs more reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def safe_mkdir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def tokenize_words(text: str) -> List[str]:
    """Simple tokenization for perturbation-rate calculations."""
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

def perturbation_rate(original: str, adversarial: str) -> float:
    """ Approximate fraction of tokens that changed. """
    orig = tokenize_words(original)
    adv = tokenize_words(adversarial)
    if not orig:
        return 0.0
    # Pad to the longer length so insertions/deletions count as changes.
    max_len = max(len(orig), len(adv))
    changes = 0
    for i in range(max_len):
        o = orig[i] if i < len(orig) else "<MISSING>"
        a = adv[i] if i < len(adv) else "<MISSING>"
        if o != a:
            changes += 1
    return changes / max_len


def lexical_overlap(original: str, adversarial: str) -> float:
    """
    Lightweight backup similarity metric if sentence-transformers is unavailable.
    Jaccard similarity over lowercased word sets.
    """
    a = set(re.findall(r"\w+", original.lower()))
    b = set(re.findall(r"\w+", adversarial.lower()))
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


class SemanticSimilarity:
    """
    Wrapper that uses SentenceTransformers cosine similarity when available.
    Falls back to lexical overlap if the embedding model is unavailable.
    """

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.mode = "lexical"
        self.model = None
        self.util = None
        try:
            from sentence_transformers import SentenceTransformer, util
            self.model = SentenceTransformer(model_name)
            self.util = util
            self.mode = "embedding"
        except Exception:
            self.mode = "lexical"

    def score(self, original: str, adversarial: str) -> float:
        if self.mode == "embedding":
            emb = self.model.encode([original, adversarial], convert_to_tensor=True)
            score = self.util.cos_sim(emb[0], emb[1]).item()
            return float(score)
        return float(lexical_overlap(original, adversarial))


@dataclass
class ExampleResult:
    attack_name: str
    attack_level: str
    example_id: int
    clean_text: str
    clean_label: int
    clean_pred: int
    clean_confidence: float
    adv_text: Optional[str]
    adv_pred: Optional[int]
    adv_confidence: Optional[float]
    attack_succeeded: bool
    perturbation_rate: Optional[float]
    semantic_similarity: Optional[float]
    notes: str = ""




## 3. Model loading

In [13]:
'''Model Loading'''

def load_classifier(model_name: str):
    """ Load a Hugging Face sequence classifier and tokenizer.
    Returns: tokenizer, model, predict_fn """
    
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    def predict(texts: List[str]) -> Tuple[np.ndarray, np.ndarray]:
        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=1)
        confs = probs.max(axis=1)
        return preds, confs

    return tokenizer, model, predict


def build_textattack_wrapper(model_name: str):
    """
    Wrap the victim model for TextAttack.
    """
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from textattack.models.wrappers import HuggingFaceModelWrapper

    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    wrapper = HuggingFaceModelWrapper(model, tokenizer)
    return wrapper




## 4. Dataset loading

In [14]:
# ------------------------------
# Dataset loading
# ------------------------------

def load_sst2_subset(num_examples: int = 50, seed: int = 42) -> pd.DataFrame:
    """
    Use GLUE SST-2 validation split. This keeps the lab aligned with a classic
    sentiment task and avoids custom dataset setup.
    """
    from datasets import load_dataset

    ds = load_dataset("glue", "sst2", split="validation")
    df = pd.DataFrame(ds)
    # Keep only the columns we need and shuffle for reproducibility.
    df = df[["sentence", "label"]].copy()
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df.iloc[:num_examples].copy()


def keep_correctly_classified_examples(df: pd.DataFrame, predict_fn, text_col: str = "sentence", label_col: str = "label") -> pd.DataFrame:
    preds, confs = predict_fn(df[text_col].tolist())
    out = df.copy()
    out["clean_pred"] = preds
    out["clean_confidence"] = confs
    out["is_correct"] = (out["clean_pred"] == out[label_col])
    return out[out["is_correct"]].reset_index(drop=True)




## 5. Attack implementations

This notebook uses:
- **DeepWordBug** for character-level perturbations
- **TextFooler** for word-level perturbations
- **CheckList-style** sentence perturbations when available
- a **paraphrase fallback** for sentence-level attacks if needed


In [15]:
# ------------------------------
# Attack implementations
# ------------------------------

def build_attack_recipe(recipe_name: str, model_wrapper):
    """
    Build a TextAttack recipe by name.
    Supported here:
      - deepwordbug
      - textfooler
      - checklist
      - bertattack
      - clare
    """
    recipe_name = recipe_name.lower()

    if recipe_name == "deepwordbug":
        from textattack.attack_recipes import DeepWordBugGao2018
        return DeepWordBugGao2018.build(model_wrapper)

    if recipe_name == "textfooler":
        from textattack.attack_recipes import TextFoolerJin2019
        return TextFoolerJin2019.build(model_wrapper)

    if recipe_name == "checklist":
        from textattack.attack_recipes import CheckList2020
        return CheckList2020.build(model_wrapper)

    if recipe_name == "bertattack":
        from textattack.attack_recipes import BERTAttackLi2020
        return BERTAttackLi2020.build(model_wrapper)

    if recipe_name == "clare":
        from textattack.attack_recipes import CLARELi2020
        return CLARELi2020.build(model_wrapper)

    raise ValueError(f"Unsupported recipe_name={recipe_name}")


def run_textattack_examples(
    examples_df: pd.DataFrame,
    model_name: str,
    recipe_name: str,
    attack_level: str,
    similarity: SemanticSimilarity,
    max_examples: Optional[int] = None,
) -> List[ExampleResult]:
    """
    Runs a TextAttack recipe one example at a time.
    One-example-at-a-time execution is slower than batch tooling, but it is easier
    to debug and easier for a report because we keep clean per-example metadata.
    """
    from textattack.shared.attacked_text import AttackedText

    wrapper = build_textattack_wrapper(model_name)
    attack = build_attack_recipe(recipe_name, wrapper)
    tokenizer, model, predict_fn = load_classifier(model_name)

    results: List[ExampleResult] = []
    run_df = examples_df.copy()
    if max_examples is not None:
        run_df = run_df.iloc[:max_examples].copy()

    for idx, row in tqdm(run_df.iterrows(), total=len(run_df), desc=f"Running {recipe_name}"):
        text = row["sentence"]
        gold = int(row["label"])
        clean_pred, clean_conf = predict_fn([text])
        clean_pred = int(clean_pred[0])
        clean_conf = float(clean_conf[0])

        try:
            initial_result = attack.attack(AttackedText(text), gold)
            # Different TextAttack versions expose result fields a little differently.
            adv_text = None
            if hasattr(initial_result, "perturbed_result") and initial_result.perturbed_result is not None:
                adv_text = initial_result.perturbed_result.attacked_text.text
            elif hasattr(initial_result, "goal_function_result") and initial_result.goal_function_result is not None:
                adv_text = initial_result.goal_function_result.attacked_text.text

            if adv_text is None:
                results.append(
                    ExampleResult(
                        attack_name=recipe_name,
                        attack_level=attack_level,
                        example_id=int(idx),
                        clean_text=text,
                        clean_label=gold,
                        clean_pred=clean_pred,
                        clean_confidence=clean_conf,
                        adv_text=None,
                        adv_pred=None,
                        adv_confidence=None,
                        attack_succeeded=False,
                        perturbation_rate=None,
                        semantic_similarity=None,
                        notes="No adversarial text produced."
                    )
                )
                continue

            adv_pred, adv_conf = predict_fn([adv_text])
            adv_pred = int(adv_pred[0])
            adv_conf = float(adv_conf[0])

            success = adv_pred != gold
            results.append(
                ExampleResult(
                    attack_name=recipe_name,
                    attack_level=attack_level,
                    example_id=int(idx),
                    clean_text=text,
                    clean_label=gold,
                    clean_pred=clean_pred,
                    clean_confidence=clean_conf,
                    adv_text=adv_text,
                    adv_pred=adv_pred,
                    adv_confidence=adv_conf,
                    attack_succeeded=success,
                    perturbation_rate=perturbation_rate(text, adv_text),
                    semantic_similarity=similarity.score(text, adv_text),
                    notes=""
                )
            )
        except Exception as e:
            results.append(
                ExampleResult(
                    attack_name=recipe_name,
                    attack_level=attack_level,
                    example_id=int(idx),
                    clean_text=text,
                    clean_label=gold,
                    clean_pred=clean_pred,
                    clean_confidence=clean_conf,
                    adv_text=None,
                    adv_pred=None,
                    adv_confidence=None,
                    attack_succeeded=False,
                    perturbation_rate=None,
                    semantic_similarity=None,
                    notes=f"Attack error: {e}"
                )
            )
    return results




In [16]:
# ------------------------------
# Sentence-level fallback attack
# ------------------------------

class ParaphraseSentenceAttack:
    """
    Fallback sentence-level attack.
    It paraphrases the full sentence, scores each candidate with the classifier, and
    keeps the best candidate that both:
        1) flips the model prediction, and
        2) preserves semantics above a threshold.

    This is not a standard TextAttack recipe, but it is a practical fallback when a
    sentence-level recipe is unavailable or unstable in a local environment.
    """

    def __init__(
        self,
        predict_fn,
        similarity: SemanticSimilarity,
        paraphrase_model_name: str = "humarin/chatgpt_paraphraser_on_T5_base",
        similarity_threshold: float = 0.80,
        num_beams: int = 10,
        num_return_sequences: int = 10,
    ):
        self.predict_fn = predict_fn
        self.similarity = similarity
        self.similarity_threshold = similarity_threshold
        self.num_beams = num_beams
        self.num_return_sequences = num_return_sequences

        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        self.pp_tokenizer = AutoTokenizer.from_pretrained(paraphrase_model_name)
        self.pp_model = AutoModelForSeq2SeqLM.from_pretrained(paraphrase_model_name)

    def paraphrase_candidates(self, text: str) -> List[str]:
        prompt = f"paraphrase: {text}"
        enc = self.pp_tokenizer([prompt], return_tensors="pt", truncation=True, padding=True)
        outputs = self.pp_model.generate(
            **enc,
            max_new_tokens=128,
            num_beams=self.num_beams,
            num_return_sequences=self.num_return_sequences,
            temperature=1.0,
        )
        candidates = self.pp_tokenizer.batch_decode(outputs, skip_special_tokens=True)
        # Deduplicate and remove unchanged text.
        uniq = []
        seen = set()
        for c in candidates:
            c = normalize_whitespace(c)
            if c and c != normalize_whitespace(text) and c not in seen:
                uniq.append(c)
                seen.add(c)
        return uniq

    def attack_example(self, text: str, gold_label: int) -> Tuple[Optional[str], Optional[int], Optional[float], str]:
        candidates = self.paraphrase_candidates(text)
        if not candidates:
            return None, None, None, "No paraphrase candidates generated."

        preds, confs = self.predict_fn(candidates)

        best_text = None
        best_pred = None
        best_conf = None
        best_score = -math.inf

        for cand, pred, conf in zip(candidates, preds, confs):
            sim = self.similarity.score(text, cand)
            if sim < self.similarity_threshold:
                continue
            # Prefer successful attacks; among them, prefer higher similarity and confidence.
            success = int(pred) != int(gold_label)
            score = (100.0 if success else 0.0) + sim + float(conf) * 0.05
            if score > best_score:
                best_score = score
                best_text = cand
                best_pred = int(pred)
                best_conf = float(conf)

        if best_text is None:
            return None, None, None, "Candidates generated, but none met the semantic threshold."

        return best_text, best_pred, best_conf, "paraphrase_fallback"

    def run(self, examples_df: pd.DataFrame) -> List[ExampleResult]:
        results = []
        preds, confs = self.predict_fn(examples_df["sentence"].tolist())
        for idx, row in tqdm(examples_df.iterrows(), total=len(examples_df), desc="Running sentence fallback"):
            text = row["sentence"]
            gold = int(row["label"])
            clean_pred = int(preds[idx])
            clean_conf = float(confs[idx])

            adv_text, adv_pred, adv_conf, note = self.attack_example(text, gold)
            if adv_text is None:
                results.append(
                    ExampleResult(
                        attack_name="paraphrase_fallback",
                        attack_level="sentence",
                        example_id=int(idx),
                        clean_text=text,
                        clean_label=gold,
                        clean_pred=clean_pred,
                        clean_confidence=clean_conf,
                        adv_text=None,
                        adv_pred=None,
                        adv_confidence=None,
                        attack_succeeded=False,
                        perturbation_rate=None,
                        semantic_similarity=None,
                        notes=note,
                    )
                )
            else:
                results.append(
                    ExampleResult(
                        attack_name="paraphrase_fallback",
                        attack_level="sentence",
                        example_id=int(idx),
                        clean_text=text,
                        clean_label=gold,
                        clean_pred=clean_pred,
                        clean_confidence=clean_conf,
                        adv_text=adv_text,
                        adv_pred=int(adv_pred),
                        adv_confidence=float(adv_conf),
                        attack_succeeded=(int(adv_pred) != gold),
                        perturbation_rate=perturbation_rate(text, adv_text),
                        semantic_similarity=self.similarity.score(text, adv_text),
                        notes=note,
                    )
                )
        return results




## 6. Reporting helpers

In [17]:
# ------------------------------
# Reporting
# ------------------------------

def to_dataframe(results: List[ExampleResult]) -> pd.DataFrame:
    return pd.DataFrame([asdict(r) for r in results])


def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate summary table required by the lab.
    """
    if df.empty:
        return pd.DataFrame()

    rows = []
    for (attack_name, attack_level), g in df.groupby(["attack_name", "attack_level"]):
        succeeded = g["attack_succeeded"].fillna(False)
        valid_sem = g["semantic_similarity"].dropna()
        valid_pert = g["perturbation_rate"].dropna()

        rows.append(
            {
                "attack_name": attack_name,
                "attack_level": attack_level,
                "num_examples": int(len(g)),
                "num_successes": int(succeeded.sum()),
                "attack_success_rate": float(succeeded.mean()),
                "avg_semantic_similarity": float(valid_sem.mean()) if len(valid_sem) else np.nan,
                "median_semantic_similarity": float(valid_sem.median()) if len(valid_sem) else np.nan,
                "avg_perturbation_rate": float(valid_pert.mean()) if len(valid_pert) else np.nan,
                "median_perturbation_rate": float(valid_pert.median()) if len(valid_pert) else np.nan,
            }
        )
    out = pd.DataFrame(rows).sort_values(["attack_level", "attack_name"]).reset_index(drop=True)
    return out


def sample_success_examples(df: pd.DataFrame, per_attack: int = 3) -> Dict[str, List[Dict[str, Any]]]:
    """
    Grab a few examples for the report.
    """
    output: Dict[str, List[Dict[str, Any]]] = {}
    if df.empty:
        return output

    for attack_name, g in df.groupby("attack_name"):
        sub = g[g["attack_succeeded"] == True].head(per_attack)
        output[attack_name] = sub[[
            "clean_text",
            "adv_text",
            "clean_label",
            "clean_pred",
            "adv_pred",
            "semantic_similarity",
            "perturbation_rate",
        ]].to_dict(orient="records")
    return output


def build_markdown_summary(
    model_name: str,
    clean_accuracy: float,
    summary_df: pd.DataFrame,
    examples: Dict[str, List[Dict[str, Any]]],
) -> str:
    """
    Human-readable summary that can be pasted into the report.
    """
    lines = []
    lines.append("# Lab 4b Auto-Generated Summary")
    lines.append("")
    lines.append("## Victim Model")
    lines.append(f"- Model: `{model_name}`")
    lines.append("- Task: Binary sentiment classification (SST-2)")
    lines.append(f"- Baseline clean accuracy on the evaluated subset: **{clean_accuracy:.3f}**")
    lines.append("")
    lines.append("## Aggregate Results")
    lines.append("")
    if summary_df.empty:
        lines.append("No results were produced.")
    else:
        lines.append(summary_df.to_markdown(index=False))
    lines.append("")
    lines.append("## Example Successful Attacks")
    lines.append("")
    if not examples:
        lines.append("No successful attack examples were found.")
    else:
        for attack_name, exs in examples.items():
            lines.append(f"### {attack_name}")
            if not exs:
                lines.append("No successful examples available.")
                lines.append("")
                continue
            for i, ex in enumerate(exs, start=1):
                lines.append(f"**Example {i}**")
                lines.append(f"- Original: {ex['clean_text']}")
                lines.append(f"- Adversarial: {ex['adv_text']}")
                lines.append(f"- clean_pred={ex['clean_pred']} | adv_pred={ex['adv_pred']}")
                if ex["semantic_similarity"] is not None:
                    lines.append(f"- semantic_similarity={ex['semantic_similarity']:.3f}")
                if ex["perturbation_rate"] is not None:
                    lines.append(f"- perturbation_rate={ex['perturbation_rate']:.3f}")
                lines.append("")
    return "\n".join(lines)


def choose_best_attack_tradeoffs(summary_df: pd.DataFrame) -> Dict[str, Optional[str]]:
    """
    Simple heuristic to help draft the discussion section.
    """
    out = {"most_effective": None, "most_stealthy": None}
    if summary_df.empty:
        return out

    eff = summary_df.sort_values(["attack_success_rate", "avg_semantic_similarity"], ascending=[False, False]).iloc[0]
    stealth = summary_df.dropna(subset=["avg_semantic_similarity", "avg_perturbation_rate"]).copy()
    if len(stealth):
        stealth["stealth_score"] = stealth["avg_semantic_similarity"] - stealth["avg_perturbation_rate"]
        stealth = stealth.sort_values("stealth_score", ascending=False).iloc[0]
        out["most_stealthy"] = f"{stealth['attack_name']} ({stealth['attack_level']})"

    out["most_effective"] = f"{eff['attack_name']} ({eff['attack_level']})"
    return out




In [18]:

# ------------------------------
# 7. Main execution cell
# ------------------------------
# Edit these settings if needed.

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
NUM_EXAMPLES = 50
SEED = 42
OUTPUT_DIR = "outputs_lab4b"
USE_SENTENCE_FALLBACK = False
SENTENCE_SIM_THRESHOLD = 0.80

set_seed(SEED)
out_dir = Path(OUTPUT_DIR)
safe_mkdir(out_dir)

print("[1/7] Loading victim classifier...")
tokenizer, model, predict_fn = load_classifier(MODEL_NAME)

print("[2/7] Loading dataset subset...")
raw_df = load_sst2_subset(num_examples=NUM_EXAMPLES, seed=SEED)

print("[3/7] Filtering to correctly classified clean examples...")
eval_df = keep_correctly_classified_examples(raw_df, predict_fn)
clean_accuracy = float((eval_df["clean_pred"] == eval_df["label"]).mean())
print(f"Clean accuracy on evaluated subset: {clean_accuracy:.4f}")

similarity = SemanticSimilarity()
all_results = []

print("[4/7] Running character-level attack...")
char_results = run_textattack_examples(
    examples_df=eval_df,
    model_name=MODEL_NAME,
    recipe_name="deepwordbug",
    attack_level="character",
    similarity=similarity,
)
all_results.extend(char_results)

print("[5/7] Running word-level attack...")
word_results = run_textattack_examples(
    examples_df=eval_df,
    model_name=MODEL_NAME,
    recipe_name="textfooler",
    attack_level="word",
    similarity=similarity,
)
all_results.extend(word_results)

print("[6/7] Running sentence-level attack...")
if USE_SENTENCE_FALLBACK:
    sentence_attack = ParaphraseSentenceAttack(
        predict_fn=predict_fn,
        similarity=similarity,
        similarity_threshold=SENTENCE_SIM_THRESHOLD,
    )
    sentence_results = sentence_attack.run(eval_df)
else:
    try:
        sentence_results = run_textattack_examples(
            examples_df=eval_df,
            model_name=MODEL_NAME,
            recipe_name="checklist",
            attack_level="sentence",
            similarity=similarity,
        )
        produced_any = any(r.adv_text is not None for r in sentence_results)
        if not produced_any:
            print("CheckList produced no adversarial texts; switching to paraphrase fallback.")
            sentence_attack = ParaphraseSentenceAttack(
                predict_fn=predict_fn,
                similarity=similarity,
                similarity_threshold=SENTENCE_SIM_THRESHOLD,
            )
            sentence_results = sentence_attack.run(eval_df)
    except Exception as e:
        print(f"Sentence-level recipe failed ({e}); switching to paraphrase fallback.")
        sentence_attack = ParaphraseSentenceAttack(
            predict_fn=predict_fn,
            similarity=similarity,
            similarity_threshold=SENTENCE_SIM_THRESHOLD,
        )
        sentence_results = sentence_attack.run(eval_df)

all_results.extend(sentence_results)

print("[7/7] Saving outputs...")
all_df = to_dataframe(all_results)
summary_df = summarize_results(all_df)
tradeoffs = choose_best_attack_tradeoffs(summary_df)
example_dict = sample_success_examples(all_df, per_attack=3)

eval_df.to_csv(out_dir / "clean_eval_examples.csv", index=False)
all_df.to_csv(out_dir / "all_attack_results.csv", index=False)
summary_df.to_csv(out_dir / "summary_results.csv", index=False)

payload = {
    "model_name": MODEL_NAME,
    "num_clean_eval_examples": int(len(eval_df)),
    "clean_accuracy_on_subset": clean_accuracy,
    "similarity_mode": similarity.mode,
    "tradeoff_summary": tradeoffs,
    "example_attacks": example_dict,
}
with open(out_dir / "results_summary.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

markdown_summary = build_markdown_summary(
    model_name=MODEL_NAME,
    clean_accuracy=clean_accuracy,
    summary_df=summary_df,
    examples=example_dict,
)
with open(out_dir / "auto_report_summary.md", "w", encoding="utf-8") as f:
    f.write(markdown_summary)

print("Done. Outputs saved to:", out_dir.resolve())
summary_df


[1/7] Loading victim classifier...


C:\Users\betha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\betha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\betha\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggi

KeyboardInterrupt: 

## 8. What to submit

After running the notebook, use these outputs:
- `outputs_lab4b/summary_results.csv` for the results table
- `outputs_lab4b/auto_report_summary.md` for drafting the report language
- `outputs_lab4b/all_attack_results.csv` for appendix evidence / attack examples

This notebook is the **code deliverable only**. The technical report and results table are separate files.
